<a href="https://colab.research.google.com/github/acoiman/pdt/blob/main/asthma_mortality/notebooks/Python/13.Asthma_Mortality_RF_CLF_SA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🥺 Sensitivity Analysis (SA) of Random Forest Classification Model



This notebook applies Global Sensitivity Analysis (GSA) to estimate the generative importance of features within a Random Forest (RF) classification model to predict the nonoccurrence (0) or occurrence (1) of asthma mortality (binary classification) by departments in Argentina, moving beyond predictive accuracy alone. While variable importance measures and local tools like SHAP quantify feature contributions to specific predictive outcomes, they do not clarify how variables structurally determine the response within the data-generating process [1]. Using Sobol indices, we explore the full input space to capture non-linear effects and decompose model variance into direct and interaction-driven components [1] . This analysis evaluates the structural dependence of the model on Population Density (PD), previously identified by SHAP as the primary driver of Asthma Mortality Occurence (AMO) across departments in Argentina for 2022. Through this generative lens, we aim to provide a more transparent and scientifically defensible explanation of the demographic and environmental factors driving asthma mortality [1] .

## 📚 Import required libraries

In [ ]:
# dataframe libraries
import pandas as pd
import numpy as np

# geospatial libraries
import geopandas as gpd
import mapclassify
from libpysal.weights import Queen
from esda.moran import Moran
from pysal.explore import esda

# plot libraries
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from matplotlib.patches import Patch
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# sklearn libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score,explained_variance_score,median_absolute_error, max_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.model_selection import ParameterSampler

# Sentivity analysis libraries
from SALib.sample import sobol as sobol_sampler
from SALib.analyze import sobol
import shap

# other libraries
import os
from joblib import Parallel, delayed
from itables import init_notebook_mode, show
from lime.lime_tabular import LimeTabularExplainer
import itertools

In [ ]:
%cd work/

In [ ]:
# Set the PROJ_LIB path
os.environ['PROJ_LIB'] = "/opt/conda/envs/gds/share/proj"

In [ ]:
 # Copy-on-Write
 pd.options.mode.copy_on_write = True

## 🆘 Helper Functions

### Functions to load and transform the dataset

In [ ]:
# path to gdf
path = "pdt/asthma_mortality/data/gpkg/data.gpkg"

In [ ]:
# Funtion to create lag variables (up to 2 years)
def create_lags(df, var, max_lag=2):
  """
  Create lagged variables (up to 2 years).

    Parameters:
    df (pd.DataFrame): Original DataFrame.
    var (str): Name of the variable to create lags for.

    Returns:
    df (pd.DataFrame): DataFrame with lagged variables.
  """

  for lag in range(1, max_lag+1):
    df[f"{var}_lag{lag}"] = df[var].shift(lag)
  return df

In [ ]:
def create_ts_long_format(path):
  """
  Create long-format time series dataset from GeoDataFrame.

    Parameters:
    path (str): Path to the GeoPackage file containing the GeoDataFrame.

    Returns:
    df_ts (pd.DataFrame): Long-format time series dataset.
  """

  # Load dataset with data per department
  gdf = gpd.read_file(path)

  # Reshape df to ts long format
  years = range(2001, 2023)
  records = []
  # iterate rows
  # exclude geometry, PD, NBUT
  for _, row in gdf.iterrows():
    iddpto = row["IDDPTO"]
    for year in years:
        records.append({
            "IDDPTO": iddpto,
            "YEAR": year,
            "AMC": row.get(f"C_{year}", np.nan), # number of Asthma Mortality C-asess
            "POP" : row.get(f"A_{year}", np.nan), # population
            "AMR": row.get(f"AMR_{year}", np.nan),
            "PM25": row.get(f"PM25_{year}", np.nan),
            "PM10": row.get(f"PM10_{year}", np.nan),
            "NBA": row.get(f"NBA_{year}", np.nan),
            "PD": row.get(f"PD_{year}", np.nan),
            "NAGRT": row.get(f"NAGRT_{year}", np.nan),
            "NNWVT": row.get(f"NNWVT_{year}", np.nan),
            "NBUT": row.get(f"NBUT_{year}", np.nan)
            })

  # create new df from list
  df_ts = pd.DataFrame(records)

  # Create lagged variables
  for var in ["NAGRT", "NNWVT", "NBUT"]:
    df_ts = create_lags(df_ts, var)

  # Drop the initial rows with NaNs due to lagging
  df_ts = df_ts.dropna().reset_index(drop=True)

  return df_ts

 ###   Function to Train Random Forest Classification Model on 2001–2021 Data and Predicting 2022 using Default Hyperparameters

In [ ]:
def train_eval_rfclass(
    df_ts: pd.DataFrame,
    features: list,
    target: str = 'AMR_BIN',
    train_end_year: int = 2021,
    test_year: int = 2022,
    random_state: int = 42
) -> str:
    """
    Trains and evaluates a Random Forest Classifier on time-series data.

    Parameters
    ----------
    df_ts : pd.DataFrame
        The input time-series dataframe.
    features : list
        A list of feature column names to be used for training.
    target : str, optional
        The name of the target variable column. Defaults to 'AMR_BIN'.
    train_end_year : int, optional
        The last year to include in the training set. Defaults to 2021.
    test_year : int, optional
        The year to use for the test set. Defaults to 2022.
    random_state : int, optional
        Control the randomness of the estimator. Defaults to 42.

    Returns
    -------
    str
        A formatted string containing accuracy, precision, recall, and F1 score.
    """

    # Training and test sets
    train_df = df_ts[df_ts['YEAR'] <= train_end_year].dropna(subset=features + [target])
    test_df = df_ts[df_ts['YEAR'] == test_year].dropna(subset=features + [target])

    # Define inputs
    X_train = train_df[features]
    y_train = train_df[target]

    X_test = test_df[features]
    y_test = test_df[target].reset_index(drop=True)

    # Train model and fit the model
    clf = RandomForestClassifier(random_state=random_state)
    clf.fit(X_train, y_train)

    # Predict using only feature columns
    y_pred = clf.predict(X_test[features])

    # Evaluate prediction
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    return (
        f"Prediction Results for {test_year}:\n"
        f"Accuracy:  {accuracy:.3f}\n"
        f"Precision: {precision:.3f}\n"
        f"Recall:    {recall:.3f}\n"
        f"F1 Score:  {f1:.3f}"
    )

### Function for Hyperparameter Tuning of the Random Forest Classification Model

In [ ]:
def tune_rfclass(
    df_ts: pd.DataFrame,
    features: list,
    param_grid: dict,
    target: str = 'AMR_BIN',
    train_end_year: int = 2021,
    test_year: int = 2022,
    n_jobs: int = -1
):
    """
    Hyperparameter tuning for Random Forest Classifier using
    training data up to train_end_year and testing on test_year.

    Parameters
    ----------
    df_ts : pd.DataFrame
        Input dataframe.

    features : list
        Predictor variables.

    param_grid : dict
        Hyperparameter grid. Example:

        {
            'n_estimators': [200, 250, 300],
            'max_depth': [10, 20, None],
            'min_samples_leaf': [1, 3, 5]
        }

    target : str, default='AMR_BIN'
        Classification target.

    train_end_year : int, default=2021
        Last year included in training.

    test_year : int, default=2022
        Year used for testing.

    n_jobs : int, default=-1
        Number of parallel workers.

    Returns
    -------
    best_result : dict
        Best parameter combination and metrics.

    results_df : pd.DataFrame
        Results for all parameter combinations.
    """

    # Split data
    train_df = df_ts[
        (df_ts['YEAR'] <= train_end_year)
    ].dropna(subset=features + [target])

    test_df = df_ts[
        (df_ts['YEAR'] == test_year)
    ].dropna(subset=features + [target])

    if train_df.empty:
        raise ValueError("No training data found after dropping NaNs.")

    if test_df.empty:
        raise ValueError("No testing data found after dropping NaNs.")

    X_train = train_df[features].values
    y_train = train_df[target].values

    X_test = test_df[features].values
    y_test = test_df[target].values

    # Generate all parameter combinations
    param_combinations = [
        dict(zip(param_grid.keys(), values))
        for values in itertools.product(*param_grid.values())
    ]

    print(f"Evaluating {len(param_combinations)} combinations")

    def evaluate_params(params):

        model = RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            n_jobs=-1,
            random_state=42
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        return {
            "params": params,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1
        }

    # Parallel evaluation
    results = Parallel(
        n_jobs=n_jobs,
        prefer="threads"
    )(
        delayed(evaluate_params)(params)
        for params in param_combinations
    )

    # Convert to dataframe
    results_df = pd.DataFrame(results)

    # Sort by F1 Score, then Accuracy (descending for both)
    results_df = results_df.sort_values(
        by=["F1 Score", "Accuracy"],
        ascending=[False, False]
    ).reset_index(drop=True)

    best_result = results_df.iloc[0].to_dict()

    print("\nBest Parameters")
    print(best_result["params"])

    print("\nPerformance")
    print(f"Accuracy : {best_result['Accuracy']:.3f}")
    print(f"Precision: {best_result['Precision']:.3f}")
    print(f"Recall   : {best_result['Recall']:.3f}")
    print(f"F1 Score : {best_result['F1 Score']:.3f}")

    return best_result, results_df

### Function to Prepare Data for SOBOL Analysis

In [ ]:
def rf_class_sobol(
    df_ts: pd.DataFrame,
    features: list,
    target: str = 'AMR_BIN',
    train_end_year: int = 2021,
    test_year: int = 2022,
) -> tuple:
    """
    Trains a Random Forest Classifier and evaluates its performance for Sobol analysis.

    Parameters
    ----------
    df_ts : pd.DataFrame
        The input time-series dataframe containing all features and the target.
    features : list
        A list of feature column names to be used for training and prediction.
    target : str, optional
        The name of the target variable column. Defaults to 'AMR_BIN'.
    train_end_year : int, optional
        The last year to include in the training set. Defaults to 2021.
    test_year : int, optional
        The year to use for the test set. Defaults to 2022.


    Returns
    -------
    tuple
        A tuple containing:
        - clf (RandomForestClassifier): The trained Random Forest classifier model.
        - metrics (dict): A dictionary of evaluation metrics (accuracy, precision, recall, F1 score).
        - X_train (pd.DataFrame): Training features.
        - X_test (pd.DataFrame): Test features.
    """

    # Training and test sets
    train_df = df_ts[df_ts['YEAR'] <= train_end_year].dropna(subset=features + [target])
    test_df = df_ts[df_ts['YEAR'] == test_year].dropna(subset=features + [target])

    # Define inputs
    X_train = train_df[features]
    y_train = train_df[target]

    X_test = test_df[features]
    y_test = test_df[target].reset_index(drop=True)

    # Train model and fit the model
    clf = RandomForestClassifier(max_depth=10,
                                 min_samples_leaf=3,
                                 min_samples_split=2,
                                 n_estimators=200,
                                 random_state=42)
    clf.fit(X_train, y_train)

    # Predict using only feature columns
    y_pred = clf.predict(X_test[features])

    # Evaluate prediction
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    metrics = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    }

    print(f"Prediction Results for {test_year}:")
    print(f"Accuracy:  {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1 Score:  {f1:.3f}")

    return clf, X_train, X_test

## 🚫 Excluding Population Density (PD) as a Predictor Variable

In this section, we evaluate the robustness of the Random Forest Classification predictions by excluding Population Density (PD). We assess how much model performance depends on this  PD predictor variable and whether predictive accuracy remains stable. This analysis examines the model’s stability and generalizability under an alternative data configuration.



In [ ]:
# apply the funtion to create long-format time series dataset from GeoDataFrame
df_ts_no_pd = create_ts_long_format(path)

In [ ]:
# Binary target
df_ts_no_pd['AMR_BIN'] = (df_ts_no_pd['AMR'] > 0).astype(int)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
df_ts_no_pd

In [ ]:
# Define features excluding PD
features_no_pd = ['PM25', "PM10", 'NBA', 'NAGRT', 'NNWVT', 'NBUT',
                'NAGRT_lag1', 'NAGRT_lag2', 'NNWVT_lag1', 'NNWVT_lag2',
                'NBUT_lag1','NBUT_lag2']

In [ ]:
# train and evaluate RF regression excluding PD with dafault hiperparameters
results_rf_no_pd = train_eval_rfclass(df_ts=df_ts_no_pd, features=features_no_pd)

In [ ]:
# print evaluation metrics
print(results_rf_no_pd)

In [ ]:
# Define the parameter grid for hyperparameter tuning
param_grid = {
    'n_estimators'     : [100, 200, 300],# Number of trees in the forest
    'max_depth'        : [None, 5, 10],  # Maximum depth of each tree
    'min_samples_split': [2, 5, 10],     # Minimum samples required to split an internal node
    'min_samples_leaf' : [1, 2, 3]       # Minimum samples required at a leaf node
}

In [ ]:
# Apply the function to perform hyperparameter tuning
best_result_no_pd, tuning_results_no_pd = tune_rfclass(
    df_ts=df_ts_no_pd,
    features=features_no_pd,
    param_grid=param_grid,
    target='AMR_BIN'
)

The Random Forest classification model assigns high feature importance to Population Density (PD) for predicting asthma mortality across departments. The most recent evaluation results are consistent with previous analyses. Excluding PD reduces Recall from 0.594 to 0.410, a 31% drop that increases false negatives. In epidemiological applications, where failing to detect true cases carries significant consequences, these results support including PD as a predictor in the classification model.

## 🥺  Global Sensitivity Analysis (GSA)

 The GSA  quantifies the individual contributions of predictors and their joint effects on model output [2]. Unlike local or one-at-a-time approaches, Sobol analysis explores the entire input space and explicitly accounts for nonlinear effects and interactions among predictors, making it particularly suitable for complex models such as Random Forest [3][4].

In [ ]:
# apply the funtion to create long-format time series dataset from GeoDataFrame
df_ts = create_ts_long_format(path)

In [ ]:
# Binary target
df_ts['AMR_BIN'] = (df_ts['AMR'] > 0).astype(int)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
df_ts

### GSA Scenario 1:  All Features Space

In [ ]:
# Define features and target
features_pd = ['PM25', 'PM10', 'NBA', 'PD', 'NAGRT', 'NNWVT', 'NBUT',
            'NAGRT_lag1', 'NAGRT_lag2', 'NNWVT_lag1', 'NNWVT_lag2',
            'NBUT_lag1','NBUT_lag2']

In [ ]:
# get trained model and data for SOBOL analysis
clf_pd, X_train_pd, X_test_pd = rf_class_sobol(
    df_ts=df_ts,
    features=features
)

In [ ]:
# Combine training and test data (if available)
all_data = pd.concat([X_train_pd, X_test_pd])

# bounds (min–max) of all data
bounds_pd = [
    [all_data[feat].min(), all_data[feat].max()]
    for feat in features_pd
]

In [ ]:
# set the Sobol problem specification
problem_pd = {
    'num_vars': len(features_pd),
    'names': features_pd,
    'bounds': bounds_pd
}

In [ ]:
# generate Sobol quasi-random samples
N = 16384  # base sample size for highly Non-Linear Models identidied by sharp
param_values_pd = sobol_sampler.sample(
    problem_pd,
    N,
    calc_second_order=True,
    seed=42
)

In [ ]:
#  convert Sobol samples into a DataFrame
X_sobol_pd = pd.DataFrame(param_values_pd, columns=features_pd)

In [ ]:
# evaluate the trained model on the Sobol-sampled input space.
# The resulting vector Y represents model output uncertainty induced by variations in the input
#Y_pd = model_output(X_sobol_pd)
Y_pd = clf_pd.predict(X_sobol_pd)

In [ ]:
# perform Sobol variance-based sensitivity analysis
sobol_results_pd = sobol.analyze(
    problem_pd,
    Y_pd,
    calc_second_order=True,
    print_to_console=False,
    seed=42
)

In [ ]:
# convert Sobol sensitivity indices into a DataFrame
sensitivity_df_pd = pd.DataFrame({
    'Feature': problem_pd['names'],
    'S1': sobol_results_pd['S1'],
    'S1_conf': sobol_results_pd['S1_conf'],
    'ST': sobol_results_pd['ST'],
    'ST_conf': sobol_results_pd['ST_conf']
}).round(2).sort_values(by='ST', ascending=False)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=False)
sensitivity_df_pd

The Sobol results indicate that **PD** is the main variance-driving predictor in the global model response. Its indices are **S1 = 0.29 (±0.07)** and **ST = 0.92 (±0.09)**.The larger gap for PD (**0.63**) indicates that a substantial part of its contribution is associated with **interactions** rather than only main effects.  $PM_{10}$/$PM_{2.5}$ (**S1 = 0.01/0.01 ± 0.01/0.02; ST = 0.40/0.32 ±0.08/0.06**)  have smaller influence  and is also mostly indirect.

Considering confidence intervals, PD remains clearly above $PM_{10}$/$PM_{2.5}$ for first-order effects (approximate S1 ranges: PD 0.22–0.36 vs $PM_{10}$/$PM_{2.5}$ 0/-0.01–0.02/0.03). For total effects, separation is larger and the intervals do not overlap (PD ~0.83–1.01; $PM_{10}$/$PM_{2.5}$ ~0.32/0.26–0.48/0.38). This suggests with high confidence that PD is the strongest **main-effect** driver, while $PM_{10}$/$PM_{2.5}$ are less influential.

Relative to SHAP (where **PD ranks first**), Sobol  **complements and refines** the interpretation. SHAP reflects contribution to individual predictions and shows PD as highly impactful at the instance level. Sobol attributes global output variance primarily to PD(especially through interactions) and secondarily to $PM_{10}$/$PM_{2.5}$. Although the two methods signal the same variable as primary, they emphasize different aspects of model behavior (local prediction influence vs global variance decomposition).


In [ ]:
# visualize second-order sensitivities (interactions)
s2_indices = []
num_vars = problem_pd['num_vars']
for i in range(num_vars):
    for j in range(i + 1, num_vars):  # Iterate only for unique pairs (i, j) where i < j
        s2_val = sobol_results_pd['S2'][i, j]
        s2_conf_val = sobol_results_pd['S2_conf'][i, j]
        feature1 = problem_pd['names'][i]
        feature2 = problem_pd['names'][j]
        s2_indices.append({
            'Feature_Pair': f"{feature1} & {feature2}",
            'S2': round(s2_val, 2),
            'S2_conf': round(s2_conf_val, 2)
        })

sensitivity_df_pd_s2 = pd.DataFrame(s2_indices).sort_values(by='S2', ascending=False)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
sensitivity_df_pd_s2

The table shows that Population Density (PD) has its larger second-order interactions with $PM_{10}$/$PM_{2.5}$ (S2=0.04/0.02) with a realative low confidence intervals of 0.05/0.05.

### GSA Scenario 2:  All Features Space and Population (POP) excluding PD

In [ ]:
# Define features excluding PD adding POP
features_pop = ['PM25', "PM10", 'POP', 'NBA', 'NAGRT', 'NNWVT', 'NBUT',
                'NAGRT_lag1', 'NAGRT_lag2', 'NNWVT_lag1', 'NNWVT_lag2',
                'NBUT_lag1','NBUT_lag2']

In [ ]:
# get trained model and data for SOBOL analysis
clf_pop, X_train_pop, X_test_pop = \
     rf_class_sobol(
         df_ts=df_ts,
         features=features_pop,
         )

In [ ]:
# Combine training and test data (if available)
all_data = pd.concat([X_train_pop, X_test_pop])

# bounds (min–max) of all data
bounds_pop = [
    [all_data[feat].min(), all_data[feat].max()]
    for feat in features_pop
]

In [ ]:
# set the Sobol problem specification
problem_pop = {
    'num_vars': len(features_pop),
    'names': features_pop,
    'bounds': bounds_pop
}

In [ ]:
# generate Sobol quasi-random samples
N = 16384  # base sample size for highly Non-Linear Models identidied by sharp
param_values_pop = sobol_sampler.sample(
    problem_pop,
    N,
    calc_second_order=True,
    seed=42
)

In [ ]:
#  convert Sobol samples into a DataFrame
X_sobol_pop = pd.DataFrame(param_values_pop, columns=features_pop)

In [ ]:
# evaluate the trained model on the Sobol-sampled input space.
# The resulting vector Y represents model output uncertainty induced by variations in the input
Y_pop = clf_pop.predict(X_sobol_pop)

In [ ]:
# perform Sobol variance-based sensitivity analysi
sobol_results_pop = sobol.analyze(
    problem_pop,
    Y_pop,
    calc_second_order=True,
    print_to_console=False,
    seed=42
)

In [ ]:
# convert Sobol sensitivity indices into a DataFrame
sensitivity_df_pop = pd.DataFrame({
    'Feature': problem_pop['names'],
    'S1': sobol_results_pop['S1'],
    'S1_conf': sobol_results_pop['S1_conf'],
    'ST': sobol_results_pop['ST'],
    'ST_conf': sobol_results_pop['ST_conf']
}).round(2).sort_values(by='ST', ascending=False)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
sensitivity_df_pop

In the absence of PD, the Sobol results indicate that **POP** is the main variance-driving predictor in the global model response. Its indices are **S1 = 0.7 (±0.07)** and **ST = 0.98 (±0.04)**, with a very larger gap (**ST−S1 = 0.28**), which supports a predominantly **interaction effect**.

### GSA Scenario 3:  All Features Space and Population (POP)

In [ ]:
# calculate correlation coefficient
round (df_ts['POP'].corr(df_ts['PD']),2)

In [ ]:
# Define features including PD and POP
features_pop_pd = ['PM25', "PM10", 'PD', 'POP', 'NBA', 'NAGRT', 'NNWVT', 'NBUT',
                'NAGRT_lag1', 'NAGRT_lag2', 'NNWVT_lag1', 'NNWVT_lag2',
                'NBUT_lag1','NBUT_lag2']

In [ ]:
# get trained model and data for SOBOL analysis
clf_pop_pd, X_train_pop_pd, X_test_pop_pd = \
    rf_class_sobol(
        df_ts=df_ts,
        features=features_pop_pd
    )

In [ ]:
# Combine training and test data (if available)
all_data = pd.concat([X_train_pop_pd, X_test_pop_pd])

# bounds (min–max) of all data
bounds_pop_pd = [
    [all_data[feat].min(), all_data[feat].max()]
    for feat in features_pop_pd
]

In [ ]:
# set the Sobol problem specification
problem_pop_pd = {
    'num_vars': len(features_pop_pd),
    'names': features_pop_pd,
    'bounds': bounds_pop_pd
}

In [ ]:
# generate Sobol quasi-random samples
N = 16384  # base sample size for highly Non-Linear Models identidied by sharp
param_values_pop_pd = sobol_sampler.sample(
    problem_pop_pd,
    N,
    calc_second_order=False,
    seed=42
)

In [ ]:
#  convert Sobol samples into a DataFrame
X_sobol_pop_pd = pd.DataFrame(param_values_pop_pd, columns=features_pop_pd)

In [ ]:
# evaluate the trained model on the Sobol-sampled input space.
# The resulting vector Y represents model output uncertainty induced by variations in the input
Y_pop_pd = clf_pop_pd.predict(X_sobol_pop_pd)

In [ ]:
# perform Sobol variance-based sensitivity analysi
sobol_results_pop_pd = sobol.analyze(
    problem_pop_pd,
    Y_pop_pd,
    calc_second_order=False,
    print_to_console=False,
    seed=42
)

In [ ]:
# convert Sobol sensitivity indices into a DataFrame
sensitivity_df_pop_pd = pd.DataFrame({
    'Feature': problem_pop_pd['names'],
    'S1': sobol_results_pop_pd['S1'],
    'S1_conf': sobol_results_pop_pd['S1_conf'],
    'ST': sobol_results_pop_pd['ST'],
    'ST_conf': sobol_results_pop_pd['ST_conf']
}).round(2).sort_values(by='ST', ascending=False)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
sensitivity_df_pop_pd

In the presence of PD, the Sobol results indicate that **POP** is the main variance-driving predictor in the global model response. Its indices are **S1 = 0.52 (±0.08)** and **ST = 1 (±0.07)**, with a very large gap (**ST−S1 = 0.48**), which supports a predominantly **interaction effect**.

We also observe that using PD along with Population, PD adds  no points of additional explained variance. This means that most of the predictive information carried by PD overlaps with information already contained in Population.

## 🚩Conclussion

The application of Global Sensitivity Analysis (GSA) provides a clearer picture of the model's feature hierarchy, especially for Population Density (PD). While local interpretation (SHAP) ranked PD as the most influential feature at the instance level, the Sobol indices reveal a identical global hierarchy.


* **Primary Drivers**. The analysis identifies PD as the dominant predictor of variance in the global model response. PD effect is predominantly interaction (S1 = 0.29, ST = 0.92), and it is the most robust main-effect driver for predicting AMR across Argentine (scenario 1).


* **The Role of Population Density (PD)**. PD remains a significant predictor. With a first-order index (S1) of 0.29 and a total index (ST) of 0.92, nearly 70% (Gap/ST*100) of PD's contribution comes from interactions with other variables rather than from a direct effect (scenario 1).

* **Decision on PD as a Predictor**. PD should be retained in the model. However, the analysis shows that much of its predictive signal overlaps with broader population metrics (scenario 2), and PD should not be viewed as the  dominant factor as SHAP might suggest (scenario 3).

In summary, PD is the primary structural driver of the Asthma Mortality occurrence/nonoccurrence in this dataset. The GSA framework reveals the generative mechanisms of the Random Forest and shifts the focus from predictive accuracy to the  generative importance of features.

##  📚 References

[1] G. Vannucci, R. Siciliano, and A. Saltelli, “Global sensitivity analysis in random forests: unveiling generative variable importance,” Stat Methods Appl, vol. 35, no.  pp. 285–317, Jun. 2026, doi: 10.1007/s10260-026-00839-y.

[2] Tosin, M., Côrtes, A.M.A., Cunha, A. (2020). A Tutorial on Sobol’ Global Sensitivity Analysis Applied to Biological Models. In: da Silva, F.A.B., Carels, N., Trindade dos Santos, M., Lopes, F.J.P. (eds) Networks in Systems Biology. Computational Biology, vol 32. Springer, Cham. https://doi.org/10.1007/978-3-030-51862-2_6

[3] Srivastava, A., Subramaniyan, A. K., &#38; Wang, L. (2017). Analytical global sensitivity analysis with Gaussian processes. <i>Ai Edam Artificial Intelligence for Engineering Design, Analysis and Manufacturing</i>, <i>31</i>(3), 235–250. https://doi.org/10.1017/S0890060417000142

[4] Ballester-Ripoll, R., Paredes, E. G., &#38; Pajarola, R. (2019). <i>Tensor Methods for Global Sensitivity Analysis</i>. 275–276. https://doi.org/10.18420/INF2019_41


